# NLP Robot Command Parser

## 0. Setup
Prepares the environment. The repository and the SCAN dataset are cloned from GitHub. Required dependencies are installed from requirements.txt. The configuration file (config.json) is loaded to set model and training parameters.

In [7]:
# Clone repo and move into it
!git clone https://github.com/PetraMicanovic/nlp-robot-command-parser.git
%cd nlp-robot-command-parser

Cloning into 'nlp-robot-command-parser'...
remote: Enumerating objects: 707, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 707 (delta 39), reused 50 (delta 17), pack-reused 633 (from 1)
Receiving objects: 100% (707/707), 490.31 KiB | 8.75 MiB/s, done.
Resolving deltas: 100% (397/397), done.
/content/nlp-robot-command-parser/nlp-robot-command-parser


In [8]:
!git clone https://github.com/brendenlake/SCAN.git data/scan

Cloning into 'data/scan'...
remote: Enumerating objects: 205, done.
remote: Total 205 (delta 0), reused 0 (delta 0), pack-reused 205 (from 1)
Receiving objects: 100% (205/205), 11.10 MiB | 12.76 MiB/s, done.
Resolving deltas: 100% (173/173), done.
Updating files: 100% (212/212), done.


In [9]:
!pip install -q -r requirements.txt

In [10]:
import json, sys, torch, random, numpy as np
sys.path.insert(0, '.')   # makes src/ importable

with open('config.json') as f:
    cfg = json.load(f)

SEED = cfg['training']['seed']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Model  : {cfg["model"]["name"]}')

Device : cuda
Model  : t5-small


## 1. Load data
Loads the SCAN dataset using the load_scan function. The data is split into training and test sets based on the configuration.

It can be loaded in either English or Serbian depending on the selected language (`cfg['data']['lang']`). When Serbian is selected, commands and actions are automatically translated.

Basic dataset informations are displayed.

In [ ]:
from src.data.load_data import load_scan

language = cfg['data']['lang'] # 'sr' or 'en'
split = cfg['data']['scan_split']

train_data, test_data = load_scan(
    split     = split,
    base_path = cfg['data']['scan_base_path'],
    lang = language,
)

print(f'Language : {language}')
print(f'Train examples : {len(train_data)}')
print(f'Test  examples : {len(test_data)}')
print(f'First example  : {train_data[0]}')

### 1.1. Dataset statistics

In [ ]:
from src.data.translate_scan import print_stats

print_stats(train_data, test_data)

## 2. Preprocessing (tokenization)

This step prepares the dataset for sequence-to-sequence training with T5. A tokenizer is loaded based on the selected model, and the raw data is converted into Hugging Face `Dataset` format.

The dataset is then tokenized by adding a task-specific prefix, encoding commands and actions, and preparing labels for training. Tokenization is applied separately to the training and test splits using a `DatasetDict`.

In [ ]:
from src.data.preprocess import get_tokenizer, to_hf_dataset, tokenize_dataset
from datasets import DatasetDict

tokenizer = get_tokenizer(cfg['model']['name'])

raw_dataset = DatasetDict({
    'train': to_hf_dataset(train_data),
    'test' : to_hf_dataset(test_data),
})

tokenized_dataset = DatasetDict({
    split: tokenize_dataset(
        raw_dataset[split], tokenizer,
        prefix         = cfg['model']['prefix'],
        max_input_len  = cfg['model']['max_input_len'],
        max_target_len = cfg['model']['max_target_len'],
    )
    for split in ('train', 'test')
})

print('Tokenization complete.')
print(tokenized_dataset)

## 3. Load pretrained T5 model

The pretrained T5 model is loaded using the configuration parameters and moved to the target device (CPU/GPU).

In [ ]:
from src.models.t5_model import load_model

model = load_model(cfg['model']['name'], DEVICE)

## 4. Training

The Seq2SeqTrainer is initialized using the model, tokenizer, tokenized dataset and training configuration. Training is then started with `trainer.train()`, which handles the full training loop, including evaluation and checkpoint saving.

After training, the final model and tokenizer are saved locally to the `checkpoints/final` directory using `save_pretrained()`.

If training is performed in Google Colab, the saved model can be copied to Google Drive for persistence. This step is optional and environment-dependent, as local training setups do not require external storage.

In [ ]:
from src.training.trainer import build_trainer

trainer = build_trainer(
    model=model,
    tokenizer=tokenizer,
    tokenized_dataset=tokenized_dataset,
    cfg=cfg,
    model_key='model',
    device_fp16=(DEVICE == 'cuda'),
)

trainer.train()

In [ ]:
from src.training.trainer import get_checkpoint_dir
# Save final model at local checkpoints/ directory
SAVE_PATH = get_checkpoint_dir(cfg, 'model')
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'Model saved to: {SAVE_PATH}')

In [ ]:
# Copy checkpoint to Google Drive
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
drive_dest = f"/content/drive/MyDrive/nlp-robot-command-parser/{SAVE_PATH}"
os.makedirs(drive_dest, exist_ok=True)
shutil.copytree(SAVE_PATH, drive_dest, dirs_exist_ok=True)
print(f"Checkpoint copied to Google Drive: {drive_dest}")

## 5. Load trained model

The trained model and tokenizer are loaded from `checkpoints/final` using `from_pretrained()`.

If running in Google Colab, Google Drive is mounted and the path is automatically redirected to the Drive location for persistent storage.

The code checks whether the model path exists and raises an error if not. The model is then moved to the selected device (`CPU` or `GPU`).

This step allows directly using a previously trained model, skipping steps 3 and 4 (model initialization and training).

In [11]:
from src.training.trainer import get_checkpoint_dir
from transformers import T5ForConditionalGeneration, T5Tokenizer
import os

LOAD_PATH = get_checkpoint_dir(cfg, 'model')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    LOAD_PATH = f"/content/drive/MyDrive/nlp-robot-command-parser/{LOAD_PATH}"
except ImportError:
    pass

if not os.path.exists(LOAD_PATH):
    raise FileNotFoundError(f'Model not found at: {LOAD_PATH}')

model = T5ForConditionalGeneration.from_pretrained(LOAD_PATH)
tokenizer = T5Tokenizer.from_pretrained(LOAD_PATH)
model = model.to(DEVICE)

print(f' Model loaded from: {LOAD_PATH}')
print(f' Device: {DEVICE}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

 Model loaded from: /content/drive/MyDrive/nlp-robot-command-parser/checkpoints/t5_small/final
 Device: cuda


## 6. Decoding examples

This section demonstrates model predictions on example inputs using constrained decoding.
The generation process is restricted to a predefined set of valid actions by masking invalid tokens.

In [ ]:
from src.models.t5_model import predict
from src.data.translate_scan import build_bad_word_ids, get_valid_actions

valid_actions = get_valid_actions(lang=language)
bad_word_ids  = build_bad_word_ids(tokenizer, valid_actions)

# Examples in Serbian (if language='sr') or English (if language='en')
examples_sr = [
    'skoci lijevo dva puta i hodaj desno',
    'trci nasuprotno desno tri puta',
    'gledaj naokolo lijevo i skoci',
]
examples_en = [
    'jump left twice and walk right',
    'run opposite right thrice',
    'look around left and jump',
]
if language == 'sr':
    examples = examples_sr
else:
    examples = examples_en

for cmd in examples:
    pred = predict(
        cmd, model, tokenizer,
        prefix         = cfg['model']['prefix'],
        max_input_len  = cfg['model']['max_input_len'],
        max_target_len = cfg['model']['max_target_len'],
        device         = DEVICE,
        num_beams      = cfg['model']['num_beams'],
        bad_word_ids   = bad_word_ids,
    )
    print(f'Command : {cmd}')
    print(f'Actions : {pred}\n')


## 7. Evaluation

This section evaluates the model using both the Hugging Face `Trainer` API and a custom evaluation pipeline.

Evaluation metrics include:
- **Exact Match (EM)** – percentage of completely correct predictions  
- **Token Accuracy** – correctness at the token level  
- **Loss** – model error on the test set (computed during Trainer evaluation)

In [ ]:
from src.training.trainer import build_trainer

trainer = build_trainer(
    model=model,
    tokenizer=tokenizer,
    tokenized_dataset=tokenized_dataset,
    cfg=cfg,
    model_key='model',
    device_fp16=(DEVICE == 'cuda'),
)

# Trainer metrics (exact match + token accuracy on full test set)
trainer_results = trainer.evaluate()
for k, v in trainer_results.items():
    print(f'  {k}: {v}')

In [ ]:
from src.evaluation.evaluation import evaluate_model, print_exact_match
from src.evaluation.save_results import save_evaluation_results, copy_results_to_drive

results = evaluate_model(
    test_data, model, tokenizer, cfg, DEVICE,
    n=cfg['data']['n_eval']
)
print_exact_match(results)

save_evaluation_results(results, split_name=split, cfg=cfg, model_key='model')
copy_results_to_drive(cfg=cfg, model_key='model')

### 7.1. Evaluation across SCAN splits

The model is evaluated on multiple SCAN splits to assess generalization across different task distributions, including:
- simple
- length
- addprim_jump
- template_around_right

For each split, the **exact match** metric is reported on a subset of the test data.

Results are collected and displayed as a summary table at the end of the evaluation loop.

In [ ]:
import pandas as pd
from src.data.load_data import load_scan
from src.evaluation.evaluation import evaluate_model
from src.evaluation.save_results import save_evaluation_results, copy_results_to_drive


splits_to_eval = ['simple', 'length', 'addprim_jump', 'addprim_turn_left', 'template_around_right','template_jump_around_right','template_opposite_right','template_right','filler_num0','filler_num1','filler_num2','filler_num3','fewshot_num8_rep1']

print('Multi-split evaluation (model trained on:', split, ')')
print('=' * 55)

rows = []
for sp in splits_to_eval:
    try:
        _, test_split = load_scan(
            split = sp,
            base_path = cfg['data']['scan_base_path'],
            lang = cfg['data']['lang'],
        )
        results = evaluate_model(test_split, model, tokenizer, cfg, DEVICE, n=200)
        print(f"  {sp:<30} exact match: {results['exact_match']:.2%}")
        save_evaluation_results(results, split_name=sp, cfg=cfg, model_key='model')
        rows.append({'Split':sp, 'Exact Match':f"{results['exact_match']:.2%}", 'N': results['n_evaluated']})
    except FileNotFoundError:
        print(f"  {sp:<30} file not found – skipping")
        rows.append({'Split':sp, 'Exact Match':'N/A', 'N': '-'})

copy_results_to_drive(cfg=cfg, model_key='model')

df = pd.DataFrame(rows)
df.index += 1
display(df.style.set_caption(f"Multi-split evaluation (trained on: {cfg['data']['scan_split']})"))

## 8. Error analysis by command length

Evaluates model performance for different sequence lengths.
The results are grouped into `buckets`, printed to the console, and saved as a JSON file in the `results` directory.

In [ ]:
from src.evaluation.evaluation import analyse_by_length, print_length_analysis
from src.evaluation.save_results import save_length_analysis, copy_results_to_drive

buckets = analyse_by_length(
    test_data, model, tokenizer, cfg, DEVICE,
    n=cfg['data']['n_error_analysis']
)
print_length_analysis(buckets)

save_length_analysis(buckets, cfg=cfg, model_key='model')
copy_results_to_drive(cfg=cfg, model_key='model')

## 9. ASR module


### 9.1. Speech synthesis and ASR demo


Generates speech from a text command using gTTS and transcribes the generated audio with Whisper. The original command, audio sample and transcription result are displayed for comparison.

In [ ]:
from src.models.asr import text_to_speech, transcribe
from IPython.display import Audio, display
from src.evaluation.save_results import save_audio_to_drive

# Use a command from test_data
test_cmd = test_data[0]["commands"]
audio_path = text_to_speech(test_cmd, filepath=cfg['asr']['audio_path'], language = cfg['asr']['tts_lang'])
save_audio_to_drive(audio_path, filename="command_demo.mp3")

print(f'Original: "{test_cmd}"')
display(Audio(audio_path))

# Without normalization - raw Whisper output
transcript_raw = transcribe(
    audio_path,
    whisper_model_name = cfg['asr']['whisper_model'],
    language = cfg['asr']['language'],
    normalize = False,
)
print(f'Raw transcript: "{transcript_raw}"')
print(f'Match: {transcript_raw == test_cmd.lower().strip()}')

# With normalization
transcript_norm = transcribe(
    audio_path,
    whisper_model_name = cfg['asr']['whisper_model'],
    language = cfg['asr']['language'],
    normalize = True,
)
print(f'Transcript: "{transcript_norm}"')
print(f'Match: {transcript_norm == test_cmd.lower().strip()}')

### 9.2.  ASR pipeline evaluation

Runs the full ASR pipeline on a subset of SCAN test commands and compares Whisper transcriptions with and without normalization. Exact-match accuracy and evaluation results are saved for further analysis.

In [ ]:
from src.models.asr import run_asr_pipeline
from src.evaluation.save_results import save_evaluation_results, copy_results_to_drive
import random

random.seed(cfg['training']['seed'])
asr_sample = random.sample(test_data, cfg['asr']['n_asr_samples'])

commands = []
for ex in asr_sample:
    commands.append(ex["commands"])

# Run full pipeline with normalization
results_norm = run_asr_pipeline(
    commands,
    audio_dir = "results/asr_audio",
    whisper_model_name = cfg['asr']['whisper_model'],
    tts_language = cfg['asr']['tts_lang'],
    asr_language = cfg['asr']['language'],
    prefix = "cmd",
)

# Run without normalization for comparison
results_raw = run_asr_pipeline(
    commands,
    audio_dir = "results/asr_audio_raw",
    whisper_model_name = cfg['asr']['whisper_model'],
    tts_language = cfg['asr']['tts_lang'],
    asr_language = cfg['asr']['language'],
    prefix = "cmd_raw",
)

n_norm = 0
wer_norm = 0.0
for r in results_norm:
    if r["match"]:
        n_norm += 1
    wer_norm += r["wer"]

n_raw = 0
wer_raw = 0.0
for r in results_raw:
    if r["match"]:
        n_raw += 1
    wer_raw += r["wer"]

n = cfg['asr']['n_asr_samples']
print(f"ASR exact match (with normalization): {n_norm / n:.2%}")
print(f"ASR WER (with normalization): {wer_norm / n:.4f}")
print(f"ASR exact match (without normalization): {n_raw  / n:.2%}")
print(f"ASR WER (without normalization): {wer_raw / n:.4f}")


# Save results
asr_results = {
    "with_normalization": {
        "exact_match": round(n_norm / n, 4),
        "n_evaluated": n,
    },
    "without_normalization": {
        "exact_match": round(n_raw / n, 4),
        "n_evaluated": n,
    },
    "per_example_normalized": results_norm,
}
save_evaluation_results(asr_results, split_name='asr', cfg=cfg, model_key='model')
copy_results_to_drive(cfg=cfg, model_key='model')

In [ ]:
# Analysis of incorrect ASR transcriptions
print("Incorrect transcriptions:")
print("-" * 60)

for r in results_norm:
    if not r["match"]:
        print(f"Original: {r['command']}")
        print(f"Transcript: {r['transcript']}")
        print()

## 10. End-to-end pipeline evaluation

Evaluates the full pipeline (Audio → Text → Actions) using the 100 pre-generated audio files from the ASR evaluation step.
The same audio files are processed with and without transcript normalization to measure the impact of ASR errors on the final action prediction.

Results are saved to `results/evaluation_pipeline.json`.

In [ ]:
from src.pipeline import RobotCommandPipeline
from src.evaluation.save_results import save_evaluation_results, copy_results_to_drive
from src.data.load_data import load_scan
import os
import random

pipeline = RobotCommandPipeline(model, tokenizer, cfg, DEVICE)

# Reproduce the same 100-command sample used in ASR evaluation
random.seed(cfg['training']['seed'])
_, test_data_for_asr = load_scan(
    split = cfg['data']['scan_split'],
    base_path = cfg['data']['scan_base_path'],
    lang = cfg['data']['lang'],
)
asr_sample = random.sample(test_data_for_asr, cfg['asr']['n_asr_samples'])

# Audio files - try local first, fall back to Drive
audio_dir_norm = 'results/asr_audio'
audio_dir_raw = 'results/asr_audio_raw'

if not os.path.exists(audio_dir_norm):
    audio_dir_norm = '/content/drive/MyDrive/nlp-robot-command-parser/results/asr_audio'
    audio_dir_raw = '/content/drive/MyDrive/nlp-robot-command-parser/results/asr_audio_raw'

correct_norm = 0
correct_raw = 0
per_example = []

for i in range(len(asr_sample)):
    filename = f'cmd_{i:04d}.mp3'
    audio_norm  = os.path.join(audio_dir_norm, filename)
    audio_raw = os.path.join(audio_dir_raw,  f"cmd_raw_{i:04d}.mp3")
    gold_actions = asr_sample[i]['actions']

    result_norm = pipeline.run_from_audio(
        audio_norm,
        gold_actions = gold_actions,
        normalize = True,
    )

    result_raw = pipeline.run_from_audio(
        audio_raw,
        gold_actions = gold_actions,
        normalize = False,
    )

    if result_norm['correct']:
        correct_norm += 1
    if result_raw['correct']:
        correct_raw += 1

    per_example.append({
        'command': asr_sample[i]['commands'],
        'gold_actions': gold_actions,
        'transcript_norm': result_norm['asr_transcript'],
        'predicted_norm': result_norm['predicted_actions'],
        'correct_norm': result_norm['correct'],
        'transcript_raw': result_raw['asr_transcript'],
        'predicted_raw': result_raw['predicted_actions'],
        'correct_raw': result_raw['correct'],
    })

n = len(asr_sample)
print('End-to-end pipeline accuracy (Audio -> Text -> Actions)')
print('=' * 55)
print(f'With normalization : {correct_norm / n:.2%}')
print(f'Without normalization : {correct_raw  / n:.2%}')

# Save results
pipeline_results = {
    'with_normalization': {'exact_match': round(correct_norm / n, 4), 'n_evaluated': n},
    'without_normalization': {'exact_match': round(correct_raw  / n, 4), 'n_evaluated': n},
    'per_example': per_example,
}
save_evaluation_results(pipeline_results, split_name='pipeline', cfg=cfg, model_key='model')
copy_results_to_drive(cfg=cfg, model_key='model')

## 10.1. Live voice demo (my own recordings)

Runs the trained t5-small pipeline on real human speech instead of gTTS-generated audio, to see how it performs on natural voice input.

Recordings are stored in `data/audio/my_voice_demo/`, with the expected commands and actions defined in `data/my_voice_commands.json`. Since t5-small was trained with `cfg['data']['lang'] == 'sr'`, these get translated to Serbian tokens before we compare them to the model's output. No TTS involved here — the audio goes straight into Whisper and then T5.

Same as the 100-sample pipeline evaluation above, each recording is run **with and without transcript normalization**, to see how much the diacritic-stripping / phonetic-alias cleanup actually helps on real speech.


In [12]:
from src.pipeline import RobotCommandPipeline
from src.data.translate_scan import translate_actions
from src.evaluation.save_results import save_evaluation_results, copy_results_to_drive
import os
import json
import pandas as pd

pipeline = RobotCommandPipeline(model, tokenizer, cfg, DEVICE)

voice_dir = 'data/audio/my_voice_demo'
commands_json_path = 'data/my_voice_commands.json'

with open(commands_json_path, 'r', encoding='utf-8') as f:
    voice_commands = json.load(f)

print(f'Loaded {len(voice_commands)} voice command entries from {commands_json_path}')

per_example = []
correct_norm, correct_raw = 0, 0

for entry in voice_commands:
    audio_path = os.path.join(voice_dir, entry['audio_file'])

    if not os.path.exists(audio_path):
        print(f"Missing file: {audio_path}")
        continue

    expected_actions_en = entry.get('output')
    expected_actions_sr = translate_actions(expected_actions_en, cfg['data']['lang'])

    result_norm = pipeline.run_from_audio(audio_path, gold_actions=expected_actions_sr, normalize=True)
    result_raw = pipeline.run_from_audio(audio_path, gold_actions=expected_actions_sr, normalize=False)

    if result_norm.get('correct') is True:
        correct_norm += 1
    if result_raw.get('correct') is True:
        correct_raw += 1

    if expected_actions_sr:
        expected_display = expected_actions_sr
    else:
        expected_display = 'N/A'

    per_example.append({
        'File': entry['audio_file'],
        'Command': entry.get('input', ''),
        'Expected actions (sr)': expected_actions_sr,
        'Transcript (norm)': result_norm['asr_transcript'],
        'Predicted (norm)': result_norm['predicted_actions'],
        'Correct (norm)': result_norm.get('correct', 'N/A'),
        'Transcript (raw)': result_raw['asr_transcript'],
        'Predicted (raw)': result_raw['predicted_actions'],
        'Correct (raw)': result_raw.get('correct', 'N/A'),
    })

n = len(per_example)
if n:
    print(f"\nProcessed {n} recordings")
    print(f"With normalization: {correct_norm}/{n} ({correct_norm/n:.0%})")
    print(f"Without normalization : {correct_raw}/{n} ({correct_raw/n:.0%})")
else:
    print('\n No recordings processed.')

df_voice = pd.DataFrame(per_example)
display(df_voice.style.set_caption('Live voice demo results — t5-small'))
if n:
    exact_match_norm = round(correct_norm / n, 4)
    exact_match_raw = round(correct_raw / n, 4)
else:
    exact_match_norm = 0
    exact_match_raw = 0

voice_demo_results = {
    'with_normalization': {'exact_match': exact_match_norm, 'n_evaluated': n},
    'without_normalization': {'exact_match': exact_match_raw, 'n_evaluated': n},
    'per_example': per_example,
}

save_evaluation_results(voice_demo_results, split_name='voice_demo', cfg=cfg, model_key='model')
copy_results_to_drive(cfg=cfg, model_key='model')

Loaded 7 voice command entries from data/my_voice_commands.json


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/


Processed 7 recordings
With normalization: 3/7 (43%)
Without normalization : 3/7 (43%)


,File,Command,Expected actions (sr),Transcript (norm),Predicted (norm),Correct (norm),Transcript (raw),Predicted (raw),Correct (raw)
0,cmd_1.mp3,gledaj desno nakon sto skocis,I_SKOCI I_OKRENI_DESNO I_GLEDAJ,gledaj desno nakon sto skocis,I_SKOCI I_OKRENI_DESNO I_GLEDAJ,True,gledaj desno nakon što skočiš.,I_SKOCI I_OKRENI_DESNO I_GLEDAJ,True
1,cmd_2.mp3,hodaj lijevo,I_OKRENI_LIJEVO I_HODAJ,hodaj lijevo,I_OKRENI_LIJEVO I_HODAJ,True,hoda i ljevo.,I_HODA I_OKRENI_LIJEVO,False
2,cmd_3.mp3,hodaj lijevo okolo,I_OKRENI_LIJEVO I_HODAJ I_OKRENI_LIJEVO I_HODAJ I_OKRENI_LIJEVO I_HODAJ I_OKRENI_LIJEVO I_HODAJ,hodaj lijevo okolo,I_OKRENI_LIJEVO I_OKRENI_LIJEVO I_OKRENI_LIJEVO I_OKRENI_LIJEVO I_OKRENI_LIJEVO I_HODAJ,False,hoda i lievo okolo.,I_HODA I_OKRENI_LIJEVO I_OKRENI_LIJEVO I_OKRENI_LIJEVO I_OKRENI_LIJEVO I_OKRENI_LIJEVO,False
3,cmd_4.mp3,hodaj suprotno od desno,I_OKRENI_DESNO I_OKRENI_DESNO I_HODAJ,hodaj suprotno od desno,I_OKRENI_DESNO I_OKRENI_DESNO I_HODAJ I_OKRENI_DESNO I_OKRENI_DESNO,False,hoda i suprotno od desno.,I_HODA I_OKRENI_DESNO I_OKRENI_DESNO I_OKRENI_DESNO,False
4,cmd_5.mp3,okreni se desno tri puta i gledaj,I_OKRENI_DESNO I_OKRENI_DESNO I_OKRENI_DESNO I_GLEDAJ,pokreni se desno tri puta i gledaj,I_OKRENI_DESNO I_OKRENI_DESNO I_OKRENI_DESNO I_OKRENI_DESNO I_GLEDAJ,False,pokreni se desno 3 puta i gledaj.,I_OKRENI_DESNO I_OKRENI_DESNO I_OKRENI_DESNO I_GLEDAJ,True
5,cmd_6.mp3,skoci tri puta,I_SKOCI I_SKOCI I_SKOCI,skoci tri puta,I_SKOCI I_SKOCI I_SKOCI,True,skoči tri puta.,I_SKOCI I_SKOCI I_SKOCI,True
6,cmd_7.mp3,trci dva puta i skoci,I_TRCI I_TRCI I_SKOCI,traci dva puta i skoci,I_TRI_TRI_TRI_SKOCI,False,traci dva puta i skoći.,I_TRI_TRI_TRI_SKOCI,False


Evaluation results saved to: results/t5-small/evaluation_voice_demo.json
Results folder copied to Google Drive: /content/drive/MyDrive/nlp-robot-command-parser/results/t5-small


## 11. HuRIC Evaluation (Serbian)

In this section, the model is evaluated on a small HuRIC subset (around 20 commands) translated into Serbian and mapped to SCAN action sequences. The dataset is first loaded from a JSON file and converted into the SCAN format expected by the evaluation pipeline. The evaluation is then performed using the existing `evaluate_model` function. Results are reported using the Exact Match metric.

Predictions are then generated for each command and shown in a table together with the expected outputs, so it’s easy to see which ones are correct and which are not.


In [ ]:
import json
import pandas as pd
from src.evaluation.evaluation import evaluate_model, print_exact_match
from src.evaluation.save_results import save_evaluation_results, copy_results_to_drive
from src.models.t5_model import predict
from src.data.translate_scan import translate_actions

with open('data/sr_huric_scan_generalization_subset_18.json', 'r', encoding='utf-8') as f:
    huric_raw = json.load(f)

# Convert HuRIC format to SCAN format
huric_test = []
for ex in huric_raw:
    huric_test.append({
        "commands": ex["input"],
        "actions": translate_actions(ex["output"], "sr"),
    })

for ex in huric_test[:3]:
    print(ex)

# Evaluate using existing function
results = evaluate_model(
    huric_test, model, tokenizer, cfg, DEVICE,
    n=len(huric_test)
)

print("HuRIC Evaluation:")
print_exact_match(results)

rows = []
for ex in huric_test:
    pred = predict(
        ex["commands"],
        model, tokenizer,
        prefix = cfg["model"]["prefix"],
        max_input_len = cfg["model"]["max_input_len"],
        max_target_len = cfg["model"]["max_target_len"],
        device = DEVICE,
        num_beams = cfg["model"]["num_beams"],
    ).strip()

    rows.append({
        "Command": ex["commands"],
        "Expected": ex["actions"],
        "Predicted": pred,
        "Correct": "correct" if pred == ex["actions"].strip() else "incorrect",
    })

# Display table
df = pd.DataFrame(rows)
df.index += 1
display(df.style.set_caption(f"HuRIC Evaluation Results (Accuracy: {results['exact_match']:.2%})"))

# Save full results with per-example predictions
huric_full_results = {
    "exact_match": results["exact_match"],
    "n_evaluated": results["n_evaluated"],
    "per_example": rows,
}
save_evaluation_results(huric_full_results, split_name='huric', cfg=cfg, model_key='model')
copy_results_to_drive(cfg=cfg, model_key='model')